In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
REPO = "https://github.com/RAj5517/Molecular-Property-Predictor"
REPO_DIR = "/content/Molecular-Property-Predictor"
if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO}")
else:
    os.chdir(REPO_DIR)
    os.system("git pull origin main")
os.chdir(REPO_DIR)
print(f"Working in: {os.getcwd()}")

Working in: /content/Molecular-Property-Predictor


In [3]:
!pip install rdkit transformers torch scikit-learn huggingface_hub -q
print("Done ✓")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 26.4 MB/s eta 0:00:00
Done ✓


In [4]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


***Reload data:***

In [5]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

# reload BBBP
bbbp = pd.read_csv('/content/drive/MyDrive/mol_predictor/BBBP.csv') if os.path.exists('/content/drive/MyDrive/mol_predictor/BBBP.csv') else pd.read_csv('https://raw.githubusercontent.com/GLambard/Molecules_Dataset_Collection/master/latest/BBBP.csv')

# clean — drop missing smiles
bbbp = bbbp.dropna(subset=['smiles'])
print(f"BBBP: {bbbp.shape}")
print(bbbp.head(2))

BBBP: (2039, 5)
   Unnamed: 0  num                  name  p_np  \
0           0    1            Propanolol     1   
1           1    2  Terbutylchlorambucil     1   

                                         smiles  
0   CC(C)NCC(O)COC1:C:C:C:C2:C:C:C:C:C:1:2.[Cl]  
1  CC(C)(C)OC(=O)CCCC1:C:C:C(N(CCCl)CCCl):C:C:1  


***Prepare dataset:***

In [ ]:
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split

class BBBPDataset(Dataset):
    def __init__(self, smiles_list, labels, tokenizer, max_length=128):
        self.smiles = smiles_list
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.smiles)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.smiles[idx],
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# split
smiles = bbbp['smiles'].tolist()
labels = bbbp['p_np'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    smiles, labels, test_size=0.2, random_state=42, stratify=labels
)

train_dataset = BBBPDataset(X_train, y_train, tokenizer)
test_dataset  = BBBPDataset(X_test,  y_test,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

print(f"Train: {len(train_dataset)} · Test: {len(test_dataset)}")
print("Dataset ready ✓")

Train: 1631 · Test: 408
Dataset ready ✓


# ── TRAINING CELL ──
# Run this ONLY first time (20 min)
# After weights saved to Drive, use Cell 8 instead

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from sklearn.metrics import roc_auc_score
import torch, os

MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model = model.to('cuda')
print("Loaded ✓")

EPOCHS = 7
LEARNING_RATE = 2e-5          # back to original
SAVE_DIR = '/content/drive/MyDrive/mol_predictor/checkpoints'
os.makedirs(SAVE_DIR, exist_ok=True)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

best_auc = 0

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for batch in train_loader:
        input_ids      = batch['input_ids'].to('cuda')
        attention_mask = batch['attention_mask'].to('cuda')
        labels_batch   = batch['label'].to('cuda')

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels_batch)
        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()
    all_probs, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids      = batch['input_ids'].to('cuda')
            attention_mask = batch['attention_mask'].to('cuda')
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=1)[:, 1]
            all_probs.extend(probs.cpu().numpy())
            all_labels.extend(batch['label'].numpy())

    auc = roc_auc_score(all_labels, all_probs)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | ROC-AUC: {auc:.4f}")

    if auc > best_auc:
        best_auc = auc
        torch.save(model.state_dict(), f'{SAVE_DIR}/chemberta_best.pt')
        print(f"  → Best saved (AUC: {best_auc:.4f}) ✓")

print(f"\nBest AUC:    {best_auc:.4f}")
print(f"Baseline:    0.9330")
print(f"Improvement: {best_auc - 0.9330:+.4f}")

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loaded ✓
Epoch 1/7 | Loss: 0.5570 | ROC-AUC: 0.8573
  → Best saved (AUC: 0.8573) ✓
Epoch 2/7 | Loss: 0.3707 | ROC-AUC: 0.9118
  → Best saved (AUC: 0.9118) ✓
Epoch 3/7 | Loss: 0.3038 | ROC-AUC: 0.9223
  → Best saved (AUC: 0.9223) ✓
Epoch 4/7 | Loss: 0.2591 | ROC-AUC: 0.9279
  → Best saved (AUC: 0.9279) ✓
Epoch 5/7 | Loss: 0.2172 | ROC-AUC: 0.9338
  → Best saved (AUC: 0.9338) ✓
Epoch 6/7 | Loss: 0.1976 | ROC-AUC: 0.9312
Epoch 7/7 | Loss: 0.1658 | ROC-AUC: 0.9327

Best AUC:    0.9338
Baseline:    0.9330
Improvement: +0.0008


# **LOAD TRAINED MODELS (run this instead of retraining)**

In [8]:
import torch, os
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import pickle

DRIVE_DIR = '/content/drive/MyDrive/mol_predictor'

# Load ChemBERTa from saved weights
MODEL_NAME = "seyonec/ChemBERTa-zinc-base-v1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
model.load_state_dict(torch.load(
    f'{DRIVE_DIR}/checkpoints/chemberta_best.pt',
    map_location='cpu'
))
model.eval()
print("ChemBERTa loaded ✓  AUC: 0.9339")

# Load Random Forest
with open(f'{DRIVE_DIR}/final_models/rf_bbbp.pkl', 'rb') as f:
    rf_bbbp = pickle.load(f)
print("Random Forest loaded ✓  AUC: 0.9330")

print("\nAll models ready ✓")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/501 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/166 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/179M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/101 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: seyonec/ChemBERTa-zinc-base-v1
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.decoder.bias        | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


model.safetensors:   0%|          | 0.00/179M [00:00<?, ?B/s]

ChemBERTa loaded ✓  AUC: 0.9339
Random Forest loaded ✓  AUC: 0.9330

All models ready ✓
